In [13]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# Configuration
RUN_NAME = "fedavg_mlp_adult-income-census_ep5_lr0.001_bs8_iid_attack_mpaf_lambda0.3"
BASE_PATH = Path("../experiments/shap_outputs") / RUN_NAME
ALPHA = 0.3  # EWMA smoothing factor (higher = more weight on recent values)

print("✓ Configuration loaded")
print(f"  Run: {RUN_NAME}")
print(f"  Path: {BASE_PATH}")
print(f"  EWMA alpha: {ALPHA}")

✓ Configuration loaded
  Run: fedavg_mlp_adult-income-census_ep5_lr0.001_bs8_iid_attack_mpaf_lambda0.3
  Path: ../experiments/shap_outputs/fedavg_mlp_adult-income-census_ep5_lr0.001_bs8_iid_attack_mpaf_lambda0.3
  EWMA alpha: 0.3


In [14]:
def load_metadata(metadata_path):
    """Load metadata from a client directory."""
    try:
        with open(metadata_path, "r") as f:
            return json.load(f)
    except:
        return {}


def load_shap_values(shap_path):
    """Load SHAP values from npz file."""
    try:
        data = np.load(shap_path, allow_pickle=True)
        # SHAP values typically stored as 'values' or similar
        keys = list(data.keys())
        if len(keys) > 0:
            return data[keys[0]]
        return None
    except:
        return None


def compute_stability_score(shap_ref, shap_curr):
    """
    Compute stability score between the current SHAP vector and a moving-average reference.
    High score = stable (honest behavior)
    Low score = unstable (suspicious behavior)
    """
    if shap_ref is None or shap_curr is None:
        return None

    v_ref = shap_ref.flatten()
    v_i = shap_curr.flatten()

    num = np.linalg.norm(v_i - v_ref)
    denom = np.linalg.norm(v_ref) + 0.1

    return 1 - num / denom
    # # Cosine similarity:
    # norm1 = np.linalg.norm(v1)
    # norm2 = np.linalg.norm(v2)

    # if norm1 < 1e-10 or norm2 < 1e-10:
    #     return 1.0

    # similarity = np.dot(v1, v2) / (norm1 * norm2)
    # # Clamp to [0, 1]
    # return max(0, min(1, (similarity + 1) / 2))


print("✓ Helper functions defined")

✓ Helper functions defined


In [15]:
# Load data for all clients across all rounds
client_data = defaultdict(dict)
round_dirs = sorted(
    [d for d in BASE_PATH.iterdir() if d.is_dir() and d.name.startswith("round_")]
)

print(f"Loading data from {len(round_dirs)} rounds...")

for round_dir in round_dirs:
    round_num = int(round_dir.name.split("_")[1])
    client_dirs = sorted(
        [d for d in round_dir.iterdir() if d.is_dir() and d.name.startswith("client_")]
    )

    for client_dir in client_dirs:
        client_id = client_dir.name

        # Load metadata
        metadata_path = client_dir / "metadata.json"
        metadata = load_metadata(metadata_path)

        # Load SHAP values
        shap_path = client_dir / "shap_values.npz"
        shap_values = load_shap_values(shap_path)

        # Store data
        if client_id not in client_data:
            client_data[client_id] = {
                "is_malicious": metadata.get(
                    "malicious", metadata.get("is_malicious", False)
                ),
                "rounds": {},
            }

        client_data[client_id]["rounds"][round_num] = {
            "shap": shap_values,
            "metadata": metadata,
        }

print(f"✓ Loaded data for {len(client_data)} clients")
print(f"  Sample client: {list(client_data.keys())[0]}")
print(f"  Malicious: {sum(1 for c in client_data.values() if c['is_malicious'])}")
print(f"  Honest: {sum(1 for c in client_data.values() if not c['is_malicious'])}")

Loading data from 21 rounds...
✓ Loaded data for 30 clients
  Sample client: client_1
  Malicious: 6
  Honest: 24


In [16]:
# Compute round-by-round stability scores
scores_data = []

for client_id, client_info in client_data.items():
    rounds = sorted(client_info["rounds"].keys())
    shap_ref = None

    for round_num in rounds:
        shap_curr = client_info["rounds"][round_num]["shap"]

        # Compute stability score vs the moving-average reference from previous rounds
        if shap_ref is not None:
            stability = compute_stability_score(shap_ref, shap_curr)
        else:
            stability = None  # No score for first round

        scores_data.append(
            {
                "client": client_id,
                "round": round_num,
                "stability": stability,
                "is_malicious": client_info["is_malicious"],
            }
        )

        if shap_ref is None:
            shap_ref = shap_curr.astype(float).copy()
        else:
            shap_ref = ALPHA * shap_curr + (1 - ALPHA) * shap_ref

# Convert to DataFrame for easier analysis
df_scores = pd.DataFrame(scores_data)
df_scores = df_scores[df_scores["stability"].notna()]  # Remove None values

print(f"✓ Computed {len(df_scores)} stability scores")
print("\nStability Score Statistics:")
print(df_scores.groupby("is_malicious")["stability"].describe())

✓ Computed 600 stability scores

Stability Score Statistics:
              count      mean       std       min       25%       50%  \
is_malicious                                                            
False         480.0  0.678456  0.201242 -0.393873  0.634474  0.739524   
True          120.0  0.365881  0.478541 -1.907074  0.169113  0.532295   

                   75%       max  
is_malicious                      
False         0.795006  0.862920  
True          0.713513  0.824202  


In [17]:
def compute_prev_round_similarity(shap_prev, shap_curr):
    if shap_prev is None or shap_curr is None:
        return None

    v_prev = shap_prev.flatten()
    v_curr = shap_curr.flatten()

    num = np.linalg.norm(v_curr - v_prev)
    denom = np.linalg.norm(v_prev) + 0.1
    return 1 - num / denom


comparison_data = []

for client_id, client_info in client_data.items():
    rounds = sorted(client_info["rounds"].keys())
    shap_ref = None

    for i, round_num in enumerate(rounds):
        shap_curr = client_info["rounds"][round_num]["shap"]

        if i > 0:
            shap_prev = client_info["rounds"][rounds[i - 1]]["shap"]
            prev_round_sim = compute_prev_round_similarity(shap_prev, shap_curr)
        else:
            prev_round_sim = None

        if shap_ref is not None:
            ref_sim = compute_stability_score(shap_ref, shap_curr)
        else:
            ref_sim = None

        comparison_data.append(
            {
                "client": client_id,
                "round": round_num,
                "prev_round_sim": prev_round_sim,
                "ref_sim": ref_sim,
                "is_malicious": client_info["is_malicious"],
            }
        )

        if shap_ref is None:
            shap_ref = shap_curr.astype(float).copy()
        else:
            shap_ref = ALPHA * shap_curr + (1 - ALPHA) * shap_ref

df_comparison = pd.DataFrame(comparison_data).dropna()

print(f"✓ Computed {len(df_comparison)} comparison scores")
print("\nMean similarity by class:")
print(df_comparison.groupby("is_malicious")[["prev_round_sim", "ref_sim"]].mean())
print("\nMean difference (prev_round_sim - ref_sim):")
print(
    df_comparison.assign(
        diff=df_comparison["prev_round_sim"] - df_comparison["ref_sim"]
    )
    .groupby("is_malicious")["diff"]
    .mean()
)

✓ Computed 600 comparison scores

Mean similarity by class:
              prev_round_sim   ref_sim
is_malicious                          
False               0.662974  0.678456
True                0.479286  0.365881

Mean difference (prev_round_sim - ref_sim):
is_malicious
False   -0.015482
True     0.113404
Name: diff, dtype: float64
